In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# Fashion-MNIST — MLP Classification

We compare two fundamentally different MLP architectures:
- **Architecture 1 — Logistic Regression**: no hidden layers (linear baseline)
- **Architecture 2 — Shallow MLP**: 1 hidden layer, 256 units
- **Architecture 3 — Deep MLP**: 3 hidden layers (512 → 256 → 128), Dropout, BatchNorm

Each architecture is evaluated with both **SGD** and **Adam** optimisers.

### Imports

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
%matplotlib inline

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.random import set_seed

print("Tensorflow version " + tf.__version__)

### Data loading & preprocessing

**Task**: Given a 28×28 grayscale image of a fashion item, predict its category.

**Classes**: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

- Flatten images to 784-dim vectors (MLPs cannot exploit spatial structure)
- Normalise pixel values: [0, 255] → [0, 1]

In [ ]:
batch_size = 128
classes    = 10
epochs     = 100

class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

X_train = X_train.reshape(60000, 784).astype('float32') / 255
X_test  = X_test.reshape(10000,  784).astype('float32') / 255

Y_train = to_categorical(y_train, classes)
Y_test  = to_categorical(y_test,  classes)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### Visualise samples

In [ ]:
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(class_names[y_train[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST — sample images', fontsize=13)
plt.tight_layout()
plt.show()

### Helper functions

In [ ]:
def plot_history(hs, epochs, metric):
    plt.style.use('dark_background')
    plt.rcParams['figure.figsize'] = [15, 8]
    plt.rcParams['font.size'] = 16
    plt.clf()
    for label, hist in hs.items():
        plt.plot(hist.history[metric],
                 label=f'{label} train {metric}', linewidth=2)
        plt.plot(hist.history[f'val_{metric}'],
                 label=f'{label} val {metric}', linewidth=2)
    x_ticks = np.arange(0, epochs + 1, max(1, epochs // 10))
    x_ticks[0] += 1
    plt.xticks(x_ticks)
    plt.ylim((0, 1))
    plt.xlabel('Epochs')
    plt.ylabel('Loss' if metric == 'loss' else 'Accuracy')
    plt.legend()
    plt.show()


def plot_confusion_matrix(model, title='Confusion Matrix'):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    cm = confusion_matrix(y_test, y_pred)
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def clean_up(model):
    K.clear_session()
    del model
    gc.collect()

### Model builder

A flexible function that builds any MLP by passing a list of hidden layer sizes.
- `hidden_units=[]` → Logistic Regression
- `hidden_units=[256]` → Shallow MLP
- `hidden_units=[512, 256, 128]` → Deep MLP

In [ ]:
def build_and_train(
        optimizer,
        hidden_units=None,
        dropout_rate=0.0,
        use_batchnorm=False,
        epochs=100,
        batch_size=128,
        callbacks=None,
        verbose=0):

    if hidden_units is None:
        hidden_units = []

    np.random.seed(1402)
    set_seed(1981)

    inp = Input(shape=(784,), name='Input')
    x = inp
    for i, units in enumerate(hidden_units):
        x = Dense(units, activation='relu',
                  kernel_initializer='glorot_uniform',
                  name=f'Hidden-{i+1}')(x)
        if use_batchnorm:
            x = BatchNormalization(name=f'BN-{i+1}')(x)
        if dropout_rate > 0:
            x = Dropout(rate=dropout_rate, name=f'Dropout-{i+1}')(x)

    out = Dense(classes, activation='softmax',
                kernel_initializer='glorot_uniform', name='Output')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    hs = model.fit(
        X_train, Y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    print('Finished training.')
    print('------------------')
    model.summary()
    return model, hs

---
## Architecture 1 — Logistic Regression (linear baseline)
Input (784) → Output (10, softmax). No hidden layers.

In [ ]:
# SGD
lr_model_sgd, lr_hs_sgd = build_and_train(
    optimizer=SGD(learning_rate=0.01), hidden_units=[],
    epochs=epochs, batch_size=batch_size
)
lr_eval_sgd = lr_model_sgd.evaluate(X_test, Y_test, verbose=1)
clean_up(lr_model_sgd)

# Adam
lr_model_adam, lr_hs_adam = build_and_train(
    optimizer=Adam(), hidden_units=[],
    epochs=epochs, batch_size=batch_size
)
lr_eval_adam = lr_model_adam.evaluate(X_test, Y_test, verbose=1)
clean_up(lr_model_adam)

In [ ]:
print('=== Logistic Regression ===')
for name, hs, ev in [('SGD', lr_hs_sgd, lr_eval_sgd), ('Adam', lr_hs_adam, lr_eval_adam)]:
    print(f'  [{name}] Train Acc: {hs.history["accuracy"][-1]:.5f}  '
          f'Val Acc: {hs.history["val_accuracy"][-1]:.5f}  '
          f'Test Acc: {ev[1]:.5f}')

plot_history({'LR-SGD': lr_hs_sgd, 'LR-Adam': lr_hs_adam}, epochs, 'loss')
plot_history({'LR-SGD': lr_hs_sgd, 'LR-Adam': lr_hs_adam}, epochs, 'accuracy')

---
## Architecture 2 — Shallow MLP
Input (784) → Dense(256, ReLU) → Output (10, softmax). One hidden layer.

In [ ]:
# SGD
shallow_model_sgd, shallow_hs_sgd = build_and_train(
    optimizer=SGD(learning_rate=0.01), hidden_units=[256],
    epochs=epochs, batch_size=batch_size
)
shallow_eval_sgd = shallow_model_sgd.evaluate(X_test, Y_test, verbose=1)
clean_up(shallow_model_sgd)

# Adam
shallow_model_adam, shallow_hs_adam = build_and_train(
    optimizer=Adam(), hidden_units=[256],
    epochs=epochs, batch_size=batch_size
)
shallow_eval_adam = shallow_model_adam.evaluate(X_test, Y_test, verbose=1)
clean_up(shallow_model_adam)

In [ ]:
print('=== Shallow MLP [256] ===')
for name, hs, ev in [('SGD', shallow_hs_sgd, shallow_eval_sgd), ('Adam', shallow_hs_adam, shallow_eval_adam)]:
    print(f'  [{name}] Train Acc: {hs.history["accuracy"][-1]:.5f}  '
          f'Val Acc: {hs.history["val_accuracy"][-1]:.5f}  '
          f'Test Acc: {ev[1]:.5f}')

plot_history({'Shallow-SGD': shallow_hs_sgd, 'Shallow-Adam': shallow_hs_adam}, epochs, 'loss')
plot_history({'Shallow-SGD': shallow_hs_sgd, 'Shallow-Adam': shallow_hs_adam}, epochs, 'accuracy')

---
## Architecture 3 — Deep MLP
Input (784) → Dense(512) → BN → Dropout(0.3) → Dense(256) → BN → Dropout(0.3) → Dense(128) → BN → Dropout(0.3) → Output (10)

Three hidden layers with BatchNormalisation and Dropout regularisation.
Early Stopping is used to avoid overfitting.

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_accuracy', patience=15, verbose=1, restore_best_weights=True
)

# SGD
deep_model_sgd, deep_hs_sgd = build_and_train(
    optimizer=SGD(learning_rate=0.01),
    hidden_units=[512, 256, 128],
    dropout_rate=0.3,
    use_batchnorm=True,
    epochs=epochs, batch_size=batch_size,
    callbacks=[early_stopping], verbose=1
)
deep_eval_sgd = deep_model_sgd.evaluate(X_test, Y_test, verbose=1)
clean_up(deep_model_sgd)

# Adam
early_stopping = EarlyStopping(
    monitor='val_accuracy', patience=15, verbose=1, restore_best_weights=True
)
deep_model_adam, deep_hs_adam = build_and_train(
    optimizer=Adam(),
    hidden_units=[512, 256, 128],
    dropout_rate=0.3,
    use_batchnorm=True,
    epochs=epochs, batch_size=batch_size,
    callbacks=[early_stopping], verbose=1
)
deep_eval_adam = deep_model_adam.evaluate(X_test, Y_test, verbose=1)

In [ ]:
print('=== Deep MLP [512, 256, 128] + BN + Dropout(0.3) ===')
for name, hs, ev in [('SGD', deep_hs_sgd, deep_eval_sgd), ('Adam', deep_hs_adam, deep_eval_adam)]:
    ep = len(hs.history['loss'])
    print(f'  [{name}] Train Acc: {hs.history["accuracy"][-1]:.5f}  '
          f'Val Acc: {hs.history["val_accuracy"][-1]:.5f}  '
          f'Test Acc: {ev[1]:.5f}  (stopped @ epoch {ep})')

ep_sgd  = len(deep_hs_sgd.history['loss'])
ep_adam = len(deep_hs_adam.history['loss'])
plot_history({'Deep-SGD': deep_hs_sgd},  ep_sgd,  'loss')
plot_history({'Deep-Adam': deep_hs_adam}, ep_adam, 'loss')
plot_history({'Deep-SGD': deep_hs_sgd, 'Deep-Adam': deep_hs_adam},
             max(ep_sgd, ep_adam), 'accuracy')

### Confusion matrix — best model (Deep MLP / Adam)

In [ ]:
plot_confusion_matrix(deep_model_adam, title='Confusion Matrix — Deep MLP (Adam)')
clean_up(deep_model_adam)

---
## Final comparison — all architectures

In [ ]:
print(f"{'Model':<42} {'Test Acc':>10} {'Test Loss':>12}")
print('-' * 66)
rows = [
    ('Logistic Regression  / SGD',  lr_eval_sgd),
    ('Logistic Regression  / Adam', lr_eval_adam),
    ('Shallow MLP [256]    / SGD',  shallow_eval_sgd),
    ('Shallow MLP [256]    / Adam', shallow_eval_adam),
    ('Deep MLP [512,256,128] / SGD',  deep_eval_sgd),
    ('Deep MLP [512,256,128] / Adam', deep_eval_adam),
]
for name, ev in rows:
    print(f"{name:<42} {ev[1]:>10.5f} {ev[0]:>12.5f}")